# D160 — Introduction to the Brazilian Olist E-commerce Dataset

The data files used in this notebook are in `C:\\data\\olist`.

Olist is a Brazilian online marketplace. It connects customers with many sellers. This public Kaggle dataset contains orders placed through Olist. It includes customers, orders, products, sellers, payments, deliveries, and reviews. Names and IDs are hidden or replaced, so the dataset can be used for learning.

Each file stores a different part of the business. First understand what one row means in a file. The orders file has one row per order. The items, payments, reviews, and geolocation files can have more than one row for the same order or location.

## What this notebook covers

This notebook explains:

- what each of the nine files contains;
- what one row means in each file;
- the meaning and data type of every column;
- primary keys, foreign keys, and table relationships;
- the dates in an order's journey;
- missing and repeated values; and
- how to join the files without counting money more than once.

## Dataset map and relationship cardinality

```text
customers (1) ── customer_id ── (1) orders
                                      │
                 ┌────────────────────┼────────────────────┐
                 │                    │                    │
            order_items (*)      payments (*)        reviews (*)
              │       │
       product_id   seller_id
              │       │
         products   sellers
              │
       category translation

customer/seller ZIP prefix ── approximate match ── geolocation (*)
```

`orders` is the main file. `order_items` lists the products inside each order. One order can have several items and several payments. If items and payments are joined directly, rows can be repeated and totals can become wrong. Add up items and payments separately for each `order_id`, and then join the totals.

The geolocation file can contain many coordinates for one ZIP prefix. Create one representative location per ZIP prefix, such as the median latitude and longitude, before joining it to customers or sellers.

## File inventory from this local copy

| File | Rows | One row represents | Key candidate |
|---|---:|---|---|
| `olist_customers_dataset.csv` | 99,441 | one order-specific customer record | `customer_id` |
| `olist_orders_dataset.csv` | 99,441 | one order | `order_id`; `customer_id` is also unique here |
| `olist_order_items_dataset.csv` | 112,650 | one numbered item in an order | (`order_id`, `order_item_id`) |
| `olist_order_payments_dataset.csv` | 103,886 | one payment event/method for an order | (`order_id`, `payment_sequential`) |
| `olist_order_reviews_dataset.csv` | 99,224 | one submitted review record | no safe single-column PK in this extract |
| `olist_products_dataset.csv` | 32,951 | one product | `product_id` |
| `olist_sellers_dataset.csv` | 3,095 | one seller | `seller_id` |
| `olist_geolocation_dataset.csv` | 1,000,163 | one coordinate observation for a ZIP prefix | no natural PK |
| `product_category_name_translation.csv` | 71 | one Portuguese-to-English category mapping | `product_category_name` |

Row counts above were measured from the CSVs in `C:\\data\\olist`.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"C:\data\olist")
csv_files = sorted(DATA_DIR.glob("*.csv"))
assert len(csv_files) == 9, f"Expected 9 CSV files, found {len(csv_files)}"
pd.DataFrame({"file": [p.name for p in csv_files], "size_mb": [round(p.stat().st_size / 1_048_576, 2) for p in csv_files]})

## 1. Customers — `olist_customers_dataset.csv`

One row represents a customer record for one order. `customer_id` is unique and connects this file to the orders file. A buyer receives a new `customer_id` for each order. `customer_unique_id` stays the same and is used to find repeat buyers.

| Column | Recommended type | Key / nullability | Value and purpose |
|---|---|---|---|
| `customer_id` | string | PK candidate; not null | anonymized order-level customer ID; FK target from orders |
| `customer_unique_id` | string | non-unique; not null | stable anonymized buyer ID for repeat-customer analysis |
| `customer_zip_code_prefix` | string/category | FK-like; not null | first five ZIP digits; preserve as an identifier |
| `customer_city` | string/category | not null | customer city in Portuguese |
| `customer_state` | string/category | not null | two-letter Brazilian state code |

**Five-record preview**

| customer_id | customer_unique_id | zip | city | state |
|---|---|---:|---|---|
| 06b8999e2fba1a1fbc88172c00ba8bc7 | 861eff4711a542e4b93843c6dd7febb0 | 14409 | franca | SP |
| 18955e83d337fd6b2def6b18a428ac77 | 290c77bc529b7ac935b93aa66c333dc3 | 9790 | sao bernardo do campo | SP |
| 4e7b3e00288586ebd08712fdd0374a03 | 060e732b5b29e8181a18229c7b0b2b5e | 1151 | sao paulo | SP |
| b2b6027bc5c5109e529d4dc6358b12c3 | 259dac757896d24d7702b9acbbff3f3c | 8775 | mogi das cruzes | SP |
| 4f2d8ab171c80ec8364f7c12e35b23ad | 345ecd01c38d18a9036ed96c73b8d066 | 13056 | campinas | SP |

## 2. Orders — `olist_orders_dataset.csv`

One row represents one order. This is the main file for following an order from purchase to delivery. Both `order_id` and `customer_id` are unique in this file.

| Column | Recommended type | Key / nullability | Value and purpose |
|---|---|---|---|
| `order_id` | string | PK candidate; not null | anonymized order identifier |
| `customer_id` | string | FK → customers; not null | links the order to its order-level customer record |
| `order_status` | category/string | not null | workflow state such as delivered, shipped, canceled |
| `order_purchase_timestamp` | datetime | not null | checkout/purchase time |
| `order_approved_at` | datetime | 160 nulls | payment/order approval time |
| `order_delivered_carrier_date` | datetime | 1,783 nulls | handoff time to logistics carrier |
| `order_delivered_customer_date` | datetime | 2,965 nulls | actual customer delivery time |
| `order_estimated_delivery_date` | datetime | not null | promised/estimated delivery date |

**Five-record preview**

| order_id | customer_id | status | purchased | approved | carrier | delivered | estimated |
|---|---|---|---|---|---|---|---|
| e481f51cbdc54678b7cc49136f2d6af7 | 9ef432eb6251297304e76186b10a928d | delivered | 2017-10-02 10:56:33 | 2017-10-02 11:07:15 | 2017-10-04 19:55:00 | 2017-10-10 21:25:13 | 2017-10-18 |
| 53cdb2fc8bc7dce0b6741e2150273451 | b0830fb4747a6c6d20dea0b8c802d7ef | delivered | 2018-07-24 20:41:37 | 2018-07-26 03:24:27 | 2018-07-26 14:31:00 | 2018-08-07 15:27:45 | 2018-08-13 |
| 47770eb9100c2d0c44946d9cf07ec65d | 41ce2a54c0b03bf3443c3d931a367089 | delivered | 2018-08-08 08:38:49 | 2018-08-08 08:55:23 | 2018-08-08 13:50:00 | 2018-08-17 18:06:29 | 2018-09-04 |
| 949d5b44dbf5de918fe9c16f97b45f8a | f88197465ea7920adcdbec7375364d82 | delivered | 2017-11-18 19:28:06 | 2017-11-18 19:45:59 | 2017-11-22 13:39:59 | 2017-12-02 00:28:42 | 2017-12-15 |
| ad21c59c0840e6cb83a9ceb5573f8159 | 8ab97904e6daea8866dbdbc4fb7aad2c | delivered | 2018-02-13 21:18:39 | 2018-02-13 22:20:29 | 2018-02-14 19:46:34 | 2018-02-16 18:17:02 | 2018-02-26 |

## 3. Order items — `olist_order_items_dataset.csv`

One row represents one item in an order. The pair (`order_id`, `order_item_id`) identifies a row. Item numbering starts again at 1 for every order.

| Column | Recommended type | Key / nullability | Value and purpose |
|---|---|---|---|
| `order_id` | string | PK part; FK → orders; not null | parent order |
| `order_item_id` | integer | PK part; not null | line sequence inside the order |
| `product_id` | string | FK → products; not null | product sold |
| `seller_id` | string | FK → sellers; not null | seller fulfilling the item |
| `shipping_limit_date` | datetime | not null | seller's deadline to hand the item to logistics |
| `price` | decimal/float | not null | item price in Brazilian reais (BRL) |
| `freight_value` | decimal/float | not null | freight allocated to the item in BRL |

**Five-record preview**

| order_id | item | product_id | seller_id | shipping limit | price | freight |
|---|---:|---|---|---|---:|---:|
| 00010242fe8c5a6d1ba2dd792cb16214 | 1 | 4244733e06e7ecb4970a6e2683c13e61 | 48436dade18ac8b2bce089ec2a041202 | 2017-09-19 09:45:35 | 58.90 | 13.29 |
| 00018f77f2f0320c557190d7a144bdd3 | 1 | e5f2d52b802189ee658865ca93d83a8f | dd7ddc04e1b6c2c614352b383efe2d36 | 2017-05-03 11:05:13 | 239.90 | 19.93 |
| 000229ec398224ef6ca0657da4fc703e | 1 | c777355d18b72b67abbeef9df44fd0fd | 5b51032eddd242adc84c38acab88f23d | 2018-01-18 14:48:30 | 199.00 | 17.87 |
| 00024acbcdf0a6daa1e931b038114c75 | 1 | 7634da152a4610f1595efa32f14722fc | 9d7a1d34a5052409006425275ba1c2b4 | 2018-08-15 10:10:18 | 12.99 | 12.79 |
| 00042b26cf59d7ce69dfabb4e55b4fd9 | 1 | ac6c3623068f30de03045865e4e10089 | df560393f3a51e74553ab94004ba5c87 | 2017-02-13 13:57:51 | 199.90 | 18.14 |

## 4. Payments — `olist_order_payments_dataset.csv`

One row represents one payment used for an order. A customer can split an order across more than one payment, so an order can have several rows.

| Column | Recommended type | Key / nullability | Value and purpose |
|---|---|---|---|
| `order_id` | string | PK part; FK → orders; not null | paid order |
| `payment_sequential` | integer | PK part; not null | payment sequence within an order |
| `payment_type` | category/string | not null | credit card, boleto, voucher, debit card, or not defined |
| `payment_installments` | integer | not null | number of installments |
| `payment_value` | decimal/float | not null | amount paid in BRL |

**Five-record preview**

| order_id | sequence | type | installments | value |
|---|---:|---|---:|---:|
| b81ef226f3fe1789b1e8b2acac839d17 | 1 | credit_card | 8 | 99.33 |
| a9810da82917af2d9aefd1278f1dcfa0 | 1 | credit_card | 1 | 24.39 |
| 25e8ea4e93396b6fa0d3dd708e76c1bd | 1 | credit_card | 1 | 65.71 |
| ba78997921bbcdc1373bb41e913ab953 | 1 | credit_card | 8 | 107.78 |
| 42fdf880ba16b47b59251dd489d4441a | 1 | credit_card | 2 | 128.45 |

## 5. Reviews — `olist_order_reviews_dataset.csv`

One row represents one review connected to an order. Some `review_id` and `order_id` values appear more than once. For that reason, neither column is a safe primary key by itself. A database version could add a new row number as a surrogate key. Review comments are in Portuguese and many are empty.

| Column | Recommended type | Key / nullability | Value and purpose |
|---|---|---|---|
| `review_id` | string | identifier; duplicates exist; not null | anonymized review ID |
| `order_id` | string | FK → orders; duplicates exist; not null | reviewed order |
| `review_score` | integer/category | not null | rating from 1 to 5 |
| `review_comment_title` | string | 87,656 nulls | optional review title |
| `review_comment_message` | string | 58,247 nulls | optional review body |
| `review_creation_date` | datetime | not null | review invitation/creation date |
| `review_answer_timestamp` | datetime | not null | time the customer answered |

**Five-record preview** (`—` means missing)

| review_id | order_id | score | title | message | created | answered |
|---|---|---:|---|---|---|---|
| 7bc2406110b926393aa56f80a40eba40 | 73fc7af87114b39712e6da79b0a377eb | 4 | — | — | 2018-01-18 | 2018-01-18 21:46:59 |
| 80e641a11e56f04c1ad469d5645fdfde | a548910a1c6147796b98fdf73dbeba33 | 5 | — | — | 2018-03-10 | 2018-03-11 03:05:13 |
| 228ce5500dc1d8e020d8d1322874b6f0 | f9e4b658b201a9f2ecdecbb34bed034b | 5 | — | — | 2018-02-17 | 2018-02-18 14:36:24 |
| e64fb393e7b32834bb789ff8bb30750e | 658677c97b385a9be170737859d3511b | 5 | — | Recebi bem antes do prazo estipulado. | 2017-04-21 | 2017-04-21 22:02:06 |
| f7c4243c7fe1938f181bec41a392bdeb | 8e6bfb81e283fa7e4f11123a3fb894f1 | 5 | — | Parabéns lojas lannister... | 2018-03-01 | 2018-03-02 10:26:53 |

## 6. Products — `olist_products_dataset.csv`

One row represents one product. Two source column names use the spelling `lenght` instead of `length`. Code must use the column names exactly as they appear in the CSV. Category names are in Portuguese; the translation file provides English names.

| Column | Recommended type | Key / nullability | Value and purpose |
|---|---|---|---|
| `product_id` | string | PK candidate; not null | anonymized product ID |
| `product_category_name` | string/category | FK → translation; 610 nulls | Portuguese category slug |
| `product_name_lenght` | nullable integer | 610 nulls | character count of product name |
| `product_description_lenght` | nullable integer | 610 nulls | character count of description |
| `product_photos_qty` | nullable integer | 610 nulls | number of listing photos |
| `product_weight_g` | nullable numeric | 2 nulls | product weight in grams |
| `product_length_cm` | nullable numeric | 2 nulls | package/product length in cm |
| `product_height_cm` | nullable numeric | 2 nulls | package/product height in cm |
| `product_width_cm` | nullable numeric | 2 nulls | package/product width in cm |

**Five-record preview**

| product_id | category | name len | desc len | photos | weight g | length | height | width |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| 1e9e8ef04dbcff4541ed26657ea517e5 | perfumaria | 40 | 287 | 1 | 225 | 16 | 10 | 14 |
| 3aa071139cb16b67ca9e5dea641aaa2f | artes | 44 | 276 | 1 | 1000 | 30 | 18 | 20 |
| 96bd76ec8810374ed1b65e291975717f | esporte_lazer | 46 | 250 | 1 | 154 | 18 | 9 | 15 |
| cef67bcfe19066a932b7673e239eb23d | bebes | 27 | 261 | 1 | 371 | 26 | 4 | 26 |
| 9dc1a7de274444849c219cff195d0b71 | utilidades_domesticas | 37 | 402 | 4 | 625 | 20 | 17 | 13 |

## 7. Sellers — `olist_sellers_dataset.csv`

One row represents one marketplace seller. The location is approximate because only the ZIP prefix is provided.

| Column | Recommended type | Key / nullability | Value and purpose |
|---|---|---|---|
| `seller_id` | string | PK candidate; not null | anonymized seller ID |
| `seller_zip_code_prefix` | string/category | FK-like; not null | first five ZIP digits |
| `seller_city` | string/category | not null | seller city |
| `seller_state` | string/category | not null | two-letter state code |

**Five-record preview**

| seller_id | zip | city | state |
|---|---:|---|---|
| 3442f8959a84dea7ee197c632cb2df15 | 13023 | campinas | SP |
| d1b65fc7debc3361ea86b5f14c68d2e2 | 13844 | mogi guacu | SP |
| ce3ad9de960102d0677a81f5d0bb7b2d | 20031 | rio de janeiro | RJ |
| c0f3eea2e14555b6faeea3dd58c1b1c3 | 4195 | sao paulo | SP |
| 51a04a8a6bdcb23deccc82b0b80742cf | 12914 | braganca paulista | SP |

## 8. Geolocation — `olist_geolocation_dataset.csv`

One row represents one recorded location for a ZIP prefix. A ZIP prefix can appear many times with slightly different coordinates or city spellings, so it is not a primary key. For a map, group the rows by ZIP prefix and use one representative point, such as the median latitude and longitude.

| Column | Recommended type | Key / nullability | Value and purpose |
|---|---|---|---|
| `geolocation_zip_code_prefix` | string/category | non-unique join field; not null | first five ZIP digits |
| `geolocation_lat` | float | not null | latitude |
| `geolocation_lng` | float | not null | longitude |
| `geolocation_city` | string/category | not null | observed city text |
| `geolocation_state` | string/category | not null | two-letter state code |

**Five-record preview**

| zip | latitude | longitude | city | state |
|---:|---:|---:|---|---|
| 1037 | -23.545621 | -46.639292 | sao paulo | SP |
| 1046 | -23.546081 | -46.644820 | sao paulo | SP |
| 1046 | -23.546129 | -46.642951 | sao paulo | SP |
| 1041 | -23.544392 | -46.639499 | sao paulo | SP |
| 1035 | -23.541578 | -46.641607 | sao paulo | SP |

## 9. Category translation — `product_category_name_translation.csv`

One row contains a Portuguese category and its English name. This file has 71 categories, but the products file contains 73 different non-empty categories. A left join can therefore leave a few products without an English category.

| Column | Recommended type | Key / nullability | Value and purpose |
|---|---|---|---|
| `product_category_name` | string/category | PK candidate; not null | Portuguese category; joins to products |
| `product_category_name_english` | string/category | unique; not null | English category label |

**Five-record preview**

| Portuguese category | English category |
|---|---|
| beleza_saude | health_beauty |
| informatica_acessorios | computers_accessories |
| automotivo | auto |
| cama_mesa_banho | bed_bath_table |
| moveis_decoracao | furniture_decor |

## Understanding the data types

A CSV file does not define data types. Pandas guesses them when it reads the file. Text and IDs usually become `object`, whole numbers become `int64`, and decimal numbers become `float64`. Dates also start as `object` unless they are parsed. A guessed type is not always the best type for analysis.

- Convert date columns with `pd.to_datetime(..., errors='coerce')`.
- Keep IDs as strings. IDs are labels, not numbers used in calculations.
- Keep ZIP prefixes as strings or categories so that leading zeros are not lost.
- Use pandas `Int64` for whole-number columns that also contain missing values.
- `float` is convenient for practice. Fixed-point decimal values are better when money must be exact.
- State, status, payment type, and category can use the pandas `category` type to save memory.

In [ ]:
# Load the orders file and convert its date columns.
order_date_columns = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv", parse_dates=order_date_columns)
orders.info()

## How this data relates to an OLTP system

OLTP means Online Transaction Processing. An e-commerce OLTP system records daily business events: a customer places an order, the order receives items and payments, a seller ships the products, and the customer may later write a review. The Olist files represent these parts of the business.

The Kaggle files are only a saved copy of historical data. They are not the live Olist database. CSV files do not enforce primary keys or foreign keys and do not contain database indexes, user accounts, inventory updates, or a full change history. These rules must be checked during analysis. For reporting, the data can later be changed into a star schema, with order items as the fact table and customer, product, seller, location, and date as dimensions.

## Keys and important points

| Dataset | Primary or unique key | Connects to | Important point |
|---|---|---|---|
| customers | PK `customer_id` | ZIP prefix → summarized geolocation | `customer_unique_id` repeats by design |
| orders | PK `order_id`; UNIQUE `customer_id` in this extract | `customer_id` → customers | lifecycle dates may be null before completion/cancellation |
| order items | PK (`order_id`, `order_item_id`) | order, product, seller IDs | multiple rows per order |
| payments | PK (`order_id`, `payment_sequential`) | `order_id` → orders | sum rows for an order total |
| reviews | use surrogate key or deduplicated composite | `order_id` → orders | review and order IDs can repeat |
| products | PK `product_id` | category → translation | source has misspelled `lenght` fields and null attributes |
| sellers | PK `seller_id` | ZIP prefix → summarized geolocation | ZIP is approximate |
| geolocation | surrogate key if persisted | none safely without aggregation | ZIP prefix is highly non-unique |
| translation | PK `product_category_name` | referenced by products | not every product category has a translation |

In [ ]:
# This function shows row counts, data types, missing values, unique values,
# and the first five rows of a CSV file.
def profile_csv(path):
    df = pd.read_csv(path)
    profile = pd.DataFrame({
        "dtype_inferred": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "nulls": df.isna().sum(),
        "distinct_non_null": df.nunique(dropna=True),
    })
    print(f"{path.name}: {len(df):,} rows × {df.shape[1]} columns")
    display(profile)
    display(df.head(5))
    return df

# Remove the first # on the next line to profile the customers file.
# customers = profile_csv(DATA_DIR / "olist_customers_dataset.csv")

## Why totals can become wrong after a join

Suppose an order has 2 item rows and 3 payment rows. Joining both files directly produces 6 rows because every item is matched with every payment. Prices and payments are then repeated. First total the items by order. Separately total the payments by order. Then join the two totals.

In [ ]:
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")

item_totals = (items.assign(item_total=items["price"] + items["freight_value"])
                    .groupby("order_id", as_index=False)
                    .agg(item_count=("order_item_id", "count"),
                         merchandise_value=("price", "sum"),
                         freight_value=("freight_value", "sum"),
                         item_total=("item_total", "sum")))
payment_totals = (payments.groupby("order_id", as_index=False)
                          .agg(payment_rows=("payment_sequential", "count"),
                               payment_value=("payment_value", "sum")))

order_finance = (orders[["order_id", "customer_id", "order_status"]]
                 .merge(item_totals, on="order_id", how="left", validate="one_to_one")
                 .merge(payment_totals, on="order_id", how="left", validate="one_to_one"))
order_finance.head()

## Practice questions

1. Count orders by status and purchase month.
2. Calculate delivery duration and compare actual versus estimated delivery dates.
3. Find repeat buyers using `customer_unique_id`, not `customer_id`.
4. Compare item totals (`price + freight`) with payment totals and investigate differences.
5. Translate product categories and rank them by revenue or item count.
6. Compare review scores for on-time and late deliveries.
7. Summarize geolocation to one representative point per ZIP prefix before mapping.

For each answer, write down what one result row represents. Check whether the join is one-to-one or one-to-many. Also note how missing values were handled. A row count is not always the same as a count of distinct orders or customers.